In [ ]:
# Schritt 1: Packages Installieren
# !pip install -U langchain langchain-openai langchain-community pypdf faiss-cpu

In [6]:
# API-key aus .env laden
from dotenv import load_dotenv
import os

load_dotenv("/Users/konstantin/Desktop/dateien/openAi/.env")

print(os.getenv("OPENAI_API_KEY")[:10])

sk-proj-V3


In [1]:
# Schritt 2: Dokumente laden
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/Users/konstantin/Desktop/dateien/openAi/Advanced_Prompting_Guidelines.pdf")
docs = loader.load()

print(f"Geladene Seiten: {len(docs)}")
print(docs[0].page_content[:500])

Geladene Seiten: 2
Erstellt für die Interview-Vorbereitung
Seite 1
 Advanced Prompting
Praktische Leitlinien für bessere, verlässlichere Prompts
1. Ziel von gutem Prompting
Ein guter Prompt reduziert Unklarheit. Er sagt dem Modell, welche Aufgabe es hat, welchen
Kontext es nutzen soll, welches Ergebnis erwartet wird und welche Grenzen gelten. Advanced
Prompting ist deshalb weniger Magie als saubere Arbeitsanweisung.
2. Die Grundstruktur eines starken Prompts

Rolle: In welcher Funktion soll das Modell antworten?



In [4]:
# Schritt 3: Text in Chunks zerlegen
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)
print(f"Anzahl Chunks: {len(chunks)}")
print(chunks[0].page_content[:300])

Anzahl Chunks: 4
Erstellt für die Interview-Vorbereitung
Seite 1
 Advanced Prompting
Praktische Leitlinien für bessere, verlässlichere Prompts
1. Ziel von gutem Prompting
Ein guter Prompt reduziert Unklarheit. Er sagt dem Modell, welche Aufgabe es hat, welchen
Kontext es nutzen soll, welches Ergebnis erwartet wird u


In [7]:
# Schritt 4: Embeddings + Vektorindex
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)

In [8]:
# Schritt 5: Retriever bauen
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [9]:
# Prompt erstellen
question = "Welche Themen werden im Dokument beschrieben?"
relevant_docs = retriever.invoke(question)

for i, doc in enumerate(relevant_docs, start=1):
    print(f"\n--- Treffer {i} ---")
    print(doc.page_content[:500])


--- Treffer 1 ---

Gib das gewünschte Ausgabeformat vor: Tabellen, Bulletpoints, JSON, Executive Summary
oder Entscheidungslogik.

Trenne Aufgabe und Material: Erst die Arbeitsanweisung, dann den Input.

Definiere Qualitätskriterien: z. B. präzise, quellenbasiert, knapp, ohne Spekulation.

Fordere Unsicherheitsmarkierung: Das Modell soll fehlende Informationen kenntlich
machen.

Nutze Beispiele sparsam: Ein gutes Beispiel kann helfen, zu viele Beispiele engen unnötig
ein.
4. Was in professionellen Settings b

--- Treffer 2 ---
Erstellt für die Interview-Vorbereitung
Seite 2

Rubrics: gib Kriterien vor, nach denen das Modell bewerten oder filtern soll.

Self-check: bitte das Modell, die eigene Antwort kurz gegen die Kriterien zu prüfen.

Prompt-Templates: standardisierte Vorlagen für wiederkehrende Aufgaben.
6. Beispiel: schwacher vs. starker Prompt
Schwach
Stark
Analysiere dieses Dokument.
Du bist Policy-Analyst. Lies das Dokument und extrahiere die drei wichtigsten Risiken. G

In [10]:
from openai import OpenAI

client = OpenAI()

context = "\n\n".join([doc.page_content for doc in relevant_docs])

prompt = f"""
Du bist ein präziser Analyst.
Beantworte die Frage nur auf Basis des bereitgestellten Kontexts.
Wenn die Information nicht im Kontext steht, sage das klar.

Frage:
{question}

Kontext:
{context}
"""

response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt
)

print(response.output_text)

Die im Dokument beschriebenen Themen sind:

- Struktur und Gestaltung von Prompts (Arbeitsanweisungen) für KI-Modelle  
- Zielsetzung und Bedeutung eines guten Prompts zur Vermeidung von Unklarheiten  
- Grundstruktur eines starken Prompts mit Rollen-, Aufgaben-, Kontext-, Format-, Kriterien- und Grenzdefinition  
- Wichtige Guidelines für gutes Prompting, z. B. Konkretheit und klare Ausgabeformate  
- Beispiele für schwache und starke Prompts  
- Spezielle Aspekte für professionelle und fortgeschrittene Prompt-Techniken  
- Leitlinien zum Umgang mit Unsicherheiten, Quellenbezug und Sensibilität  
- Fehler, die häufig beim Prompting gemacht werden  
- Ein universelles Template für die Erstellung von Prompts  
- Anwendungsszenarien wie Interview-Vorbereitung und RAG-Systeme  

Diese Themen ergeben sich ausschließlich aus dem vorliegenden Dokumenttext. Keine Informationen außerhalb dieses Kontexts sind hinzugefügt.
